In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import math
import copy
import h5py

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")
print(f"Using device: {device}")

In [ ]:
MAT_PATH = 'MetaLearning_Dataset.mat'

# Hyperparameters

# Base T-LSTM
P, G, Fh, A = 20, 15, 3, 2
channel_dim  = 18
d_model      = 64
nhead        = 8
hidden_size  = 64

# Meta-learning
EPOCHS_PER_SHOT  = 40
sample_sizes     = [50, 800]       
num_trajectories = 40
LAMBDA           = 1.0             
META_EPOCHS      = 200

In [ ]:
# Data Loading

def load_mat_chunks(mat_path):
    def extract_chunks(f, cell_key):
        chunks =[]
        # Safely handle missing datasets
        if cell_key not in f:
            return chunks
            
        cell = f[cell_key]
        refs = cell[()].flatten()
        for ref in refs:
            grp = f[ref]
            H_ds = grp['H']
            if H_ds.dtype.names and 'real' in H_ds.dtype.names:
                H_real = H_ds['real'][()]
                H_imag = H_ds['imag'][()]
            else:
                H_real = H_ds[()]
                H_imag = np.zeros_like(H_real)
            H_r = H_real[0, :, :]
            H_i = H_imag[0, :, :]
            data = np.concatenate([H_r, H_i], axis=1).astype(np.float32)
            chunks.append(data)
        return chunks

    with h5py.File(mat_path, 'r') as f:
        head_chunks = extract_chunks(f, 'Dataset_Head')
        body_chunks = extract_chunks(f, 'Dataset_Body')
        tail_chunks = extract_chunks(f, 'Dataset_Tail')

    print(f"Loaded {len(head_chunks)} head, {len(body_chunks)} body, "
          f"and {len(tail_chunks)} tail chunks from {mat_path}")
    return head_chunks, body_chunks, tail_chunks

In [ ]:
# Time Feature Encoding

def generate_periodic_features(length, periods=(24, 168, 720)):
    i = np.arange(length, dtype=np.float32)
    features =[]
    for period in periods:
        angle = 2 * np.pi * i / period
        features.append(np.sin(angle))
        features.append(np.cos(angle))
    return np.column_stack(features)

In [ ]:
# Dataset

class TimeDataset(Dataset):
    def __init__(self, raw_data, P, G, Fh, A, periods=(24, 168, 720),
                 mean=None, std=None):
        self.P  = P
        self.G  = G
        self.Fh = Fh
        self.A  = A
        self.periods = periods

        raw = raw_data.astype(np.float32)
        self.channel_dim = raw.shape[1]

        self.mean = mean if mean is not None else raw.mean(axis=0)
        self.std  = std  if std  is not None else raw.std(axis=0)
        self.data = ((raw - self.mean) / (self.std + 1e-8)).astype(np.float32)

        self.periodic_features = generate_periodic_features(
            len(raw), periods).astype(np.float32)
        self.num_periodic = self.periodic_features.shape[1]

        self.data_dim = self.channel_dim + self.num_periodic + 1

    def __len__(self):
        return len(self.data) - self.P - self.Fh + 1

    def __getitem__(self, idx):
        P, G, Fh = self.P, self.G, self.Fh

        enc_data      = self.data[idx : idx + P]
        enc_periodic  = self.periodic_features[idx : idx + P]
        enc_time_prog = (np.arange(P, dtype=np.float32) / P).reshape(-1, 1)
        encoder_input = np.hstack([enc_data, enc_periodic, enc_time_prog])

        target = self.data[idx + P : idx + P + Fh]

        dec_data_recent = self.data[idx + P - G : idx + P]
        dec_data_zeros  = np.zeros((Fh, self.channel_dim), dtype=np.float32)
        dec_data        = np.vstack([dec_data_recent, dec_data_zeros])

        dec_periodic  = self.periodic_features[idx + P - G : idx + P + Fh]
        dec_time_prog = (np.arange(P - G, P + Fh,
                                   dtype=np.float32) / P).reshape(-1, 1)
        decoder_input = np.hstack([dec_data, dec_periodic, dec_time_prog])

        return (
            torch.as_tensor(encoder_input, dtype=torch.float32),
            torch.as_tensor(decoder_input, dtype=torch.float32),
            torch.as_tensor(target,        dtype=torch.float32),
        )

In [ ]:
# Model components

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, nhead, d_ff=256, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.dropout1(attn_out))
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, attn_layers, data_dim, d_model, norm_layer=None):
        super().__init__()
        self.input_proj  = nn.Linear(data_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.attn_layers = nn.ModuleList(attn_layers)
        self.norm        = norm_layer

    def forward(self, x):
        x = self.pos_encoder(self.input_proj(x))
        for layer in self.attn_layers:
            x = layer(x)
        if self.norm is not None:
            x = self.norm(x)
        return x

def generate_square_subsequent_mask(sz, device):
    return torch.triu(
        torch.ones(sz, sz, device=device) * float('-inf'), diagonal=1)

class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, nhead, d_ff=256, dropout=0.1):
        super().__init__()
        self.masked_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.cross_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.ffn      = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm3    = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None):
        _x, _ = self.masked_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + self.dropout1(_x))
        _x, _ = self.cross_attn(x, memory, memory)
        x = self.norm2(x + self.dropout2(_x))
        x = self.norm3(x + self.dropout3(self.ffn(x)))
        return x

class Decoder(nn.Module):
    def __init__(self, layers, data_dim, d_model, channel_dim,
                 hidden_size=64, norm_layer=None):
        super().__init__()
        self.d_model     = d_model
        self.channel_dim = channel_dim
        self.input_proj  = nn.Linear(data_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.layers      = nn.ModuleList(layers)
        self.norm        = norm_layer
        self.lstm = nn.LSTM(
            input_size=channel_dim, hidden_size=hidden_size,
            num_layers=2, batch_first=True, dropout=0.1)
        self.final_norm = nn.LayerNorm(d_model + hidden_size)  # ADDED
        self.fc = nn.Sequential(
            nn.Linear(d_model + hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, channel_dim),
        )

    def forward(self, data, memory, original_seq):
        seq_len  = data.size(1)
        tgt_mask = generate_square_subsequent_mask(seq_len, data.device)
        x = self.pos_encoder(self.input_proj(data))
        for layer in self.layers:
            x = layer(x, memory, tgt_mask)
        if self.norm is not None:
            x = self.norm(x)
        lstm_out, _ = self.lstm(original_seq)
        combined    = torch.cat([x, lstm_out], dim=-1)
        combined    = self.final_norm(combined)              # ADDED
        return self.fc(combined)

In [ ]:
# Loss and evaluation

def nmse_loss(pred, target):
    numerator   = torch.sum((pred - target) ** 2, dim=(1, 2))
    denominator = torch.sum(target ** 2,           dim=(1, 2)) + 1e-8
    return torch.mean(numerator / denominator)

def evaluate_model(encoder, decoder, loader, device, Fh, channel_dim):
    encoder.eval()
    decoder.eval()
    total_loss = 0.0
    with torch.no_grad():
        for enc_in, dec_in, target in loader:
            enc_in  = enc_in.to(device)
            dec_in  = dec_in.to(device)
            target  = target.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            pred = decoder(dec_in, encoder(enc_in), original_seq)
            total_loss += nmse_loss(pred[:, -Fh:, :], target).item()
    return total_loss / len(loader)

In [ ]:
# Weight extraction / injection

def extract_weights(decoder):
    w = decoder.fc[3].weight.detach().view(-1)
    b = decoder.fc[3].bias.detach().view(-1)
    return torch.cat([w, b]).unsqueeze(0)

def inject_weights(decoder, theta_flat, in_features=64, channel_dim=18):
    w_size = in_features * channel_dim
    w = theta_flat[0, :w_size].view(channel_dim, in_features)
    b = theta_flat[0, w_size:]
    decoder.fc[3].weight.data.copy_(w)
    decoder.fc[3].bias.data.copy_(b)
    return decoder

def decoder_forward_features(decoder, dec_input, enc_output, original_seq):
    seq_len  = dec_input.size(1)
    tgt_mask = generate_square_subsequent_mask(seq_len, dec_input.device)
    x = decoder.pos_encoder(decoder.input_proj(dec_input))
    for layer in decoder.layers:
        x = layer(x, enc_output, tgt_mask)
    if decoder.norm is not None:
        x = decoder.norm(x)
    lstm_out, _ = decoder.lstm(original_seq)
    combined    = torch.cat([x, lstm_out], dim=-1)
    combined    = decoder.final_norm(combined)               # ADDED
    return decoder.fc[:3](combined)

In [ ]:
# MetaModelNet

class MetaResidualBlock(nn.Module):
    def __init__(self, param_dim, leaky_slope=0.01):
        super().__init__()
        self.norm1 = nn.LayerNorm(param_dim)
        self.act1  = nn.LeakyReLU(leaky_slope)
        self.fc1   = nn.Linear(param_dim, param_dim)
        self.norm2 = nn.LayerNorm(param_dim)
        self.act2  = nn.LeakyReLU(leaky_slope)
        self.fc2   = nn.Linear(param_dim, param_dim)

    def forward(self, theta):
        out = self.fc1(self.act1(self.norm1(theta)))
        out = self.fc2(self.act2(self.norm2(out)))
        return theta + out

class MetaModelNet(nn.Module):
    def __init__(self, param_dim, num_blocks):
        super().__init__()
        self.num_blocks = num_blocks
        self.blocks = nn.ModuleList([MetaResidualBlock(param_dim) for _ in range(num_blocks)])

    def forward(self, theta, start_block_idx=0):
        out = theta
        for i in range(start_block_idx, self.num_blocks):
            out = self.blocks[i](out)
        return out

In [ ]:
# k-shot weight generation

def get_k_shot_weights(base_encoder, base_decoder, dataset,
                       subset_size, epochs_per_shot, device, channel_dim, Fh):
    enc_clone = copy.deepcopy(base_encoder).to(device)
    dec_clone = copy.deepcopy(base_decoder).to(device)

    for param in dec_clone.parameters():
        param.requires_grad = False
        
    for param in dec_clone.fc[3].parameters():
        param.requires_grad = True

    indices    = np.random.choice(len(dataset),
                                  min(subset_size, len(dataset)), replace=False)
    sub_loader = DataLoader(Subset(dataset, indices), batch_size=32, shuffle=True)
    optimizer  = torch.optim.Adam(dec_clone.fc[3].parameters(), lr=5e-3, weight_decay=1e-4)  # CHANGED

    enc_clone.eval()
    dec_clone.eval() 

    for _ in range(epochs_per_shot):
        for enc_in, dec_in, target in sub_loader:
            enc_in  = enc_in.to(device)
            dec_in  = dec_in.to(device)
            target  = target.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            
            optimizer.zero_grad()
            with torch.no_grad():
                encoded = enc_clone(enc_in)
                
            pred = dec_clone(dec_in, encoded, original_seq)
            loss = nmse_loss(pred[:, -Fh:, :], target)
            loss.backward()
            optimizer.step()

    return extract_weights(dec_clone)

In [ ]:
# Main script execution

# Load MATLAB data
head_chunks, body_chunks, tail_chunks = load_mat_chunks(MAT_PATH)

def chunk_is_valid(chunk, zero_threshold=0.03):
    zero_rows = np.all(chunk == 0, axis=1)
    return zero_rows.mean() < zero_threshold

head_chunks = [c for c in head_chunks if chunk_is_valid(c)]
body_chunks = [c for c in body_chunks if chunk_is_valid(c)]
tail_chunks = [c for c in tail_chunks if chunk_is_valid(c)]
print(f"After filtering: {len(head_chunks)} head, {len(body_chunks)} body, {len(tail_chunks)} tail chunks")

# Build datasets
# Pre-train on Head + Body
all_source = np.concatenate(head_chunks + body_chunks, axis=0)
n          = len(all_source)
n_train    = int(0.70 * n)
n_val      = int(0.15 * n)

train_raw = all_source[:n_train]
val_raw   = all_source[n_train : n_train + n_val]
test_raw  = all_source[n_train + n_val:]

# Global datasets for pre-training Base T-LSTM
train_dataset = TimeDataset(train_raw, P, G, Fh, A)
val_dataset   = TimeDataset(val_raw,   P, G, Fh, A,
                            mean=train_dataset.mean, std=train_dataset.std)
test_dataset  = TimeDataset(test_raw,  P, G, Fh, A,
                            mean=train_dataset.mean, std=train_dataset.std)

# Per-task datasets for meta-learning
head_datasets = [TimeDataset(chunk, P, G, Fh, A) for chunk in head_chunks]
body_datasets = [TimeDataset(chunk, P, G, Fh, A) for chunk in body_chunks]

# Split tail into meta-train (trajectory generation) and eval (held-out)
np.random.seed(42)
tail_indices = np.random.permutation(len(tail_chunks))
tail_train_idx = tail_indices[:30]
tail_eval_idx  = tail_indices[30:]

tail_train_chunks = [tail_chunks[i] for i in tail_train_idx]
tail_eval_chunks  = [tail_chunks[i] for i in tail_eval_idx]

tail_train_datasets = [TimeDataset(chunk, P, G, Fh, A) for chunk in tail_train_chunks]
tail_eval_datasets  = [TimeDataset(chunk, P, G, Fh, A) for chunk in tail_eval_chunks]

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=512, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False)

data_dim = train_dataset.data_dim
print(f"Data dim: {data_dim} | Channel dim: {channel_dim} | "
      f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | "
      f"Test: {len(test_dataset)} | "
      f"Tail train: {len(tail_train_datasets)} | Tail eval: {len(tail_eval_datasets)}")

# Build base T-LSTM 
enc_blocks = [TransformerEncoderBlock(d_model, nhead) for _ in range(2)]
base_encoder = TransformerEncoder(
    enc_blocks, data_dim, d_model, nn.LayerNorm(d_model)).to(device)

dec_blocks = [TransformerDecoderBlock(d_model, nhead) for _ in range(2)]
base_decoder = Decoder(
    dec_blocks, data_dim, d_model, channel_dim, hidden_size).to(device)

print("Base T-LSTM initialised.")

In [ ]:
PRE_TRAIN_EPOCHS = 500
EARLY_STOP_PATIENCE = 20
best_val = float('inf')
patience_counter = 0

optimizer = torch.optim.Adam(
    list(base_encoder.parameters()) + list(base_decoder.parameters()), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5)

print("Pre-training base T-LSTM on head+body data...")
for epoch in range(1, PRE_TRAIN_EPOCHS + 1):
    base_encoder.train()
    base_decoder.train()
    train_loss = 0.0

    for enc_in, dec_in, target in train_loader:
        enc_in  = enc_in.to(device)
        dec_in  = dec_in.to(device)
        target  = target.to(device)
        original_seq = dec_in[:, :, :channel_dim]
        
        optimizer.zero_grad()
        encoded = base_encoder(enc_in)
        pred    = base_decoder(dec_in, encoded, original_seq)
        loss    = nmse_loss(pred[:, -Fh:, :], target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    val_loss = evaluate_model(
        base_encoder, base_decoder, val_loader, device, Fh, channel_dim)
    scheduler.step(val_loss)

    if val_loss < best_val - 1e-5:
        best_val = val_loss
        patience_counter = 0
        torch.save({'encoder': base_encoder.state_dict(),
                    'decoder': base_decoder.state_dict()}, 'best_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val: {best_val:.4f}")
            break

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Train: {train_loss/len(train_loader):.4f} | Val: {val_loss:.4f}")

ckpt = torch.load('best_model.pt', weights_only=True)
base_encoder.load_state_dict(ckpt['encoder'])
base_decoder.load_state_dict(ckpt['decoder'])
print("Pre-training complete. Best weights restored.")

In [ ]:
ckpt = torch.load('best_model.pt', weights_only=True)
base_encoder.load_state_dict(ckpt['encoder'])
base_decoder.load_state_dict(ckpt['decoder'])
print("Pre-training skipped. Best weights restored.")

In [ ]:
# Peek at the first few weights of the encoder input projection
print("First 5 encoder input_proj weights:",
      base_encoder.input_proj.weight.data.flatten()[:5].cpu().numpy())

In [ ]:
base_encoder.eval()
base_decoder.eval()

sample_enc, sample_dec, sample_tgt = next(iter(DataLoader(
    head_datasets[0], batch_size=4, shuffle=False)))
sample_enc = sample_enc.to(device)
sample_dec = sample_dec.to(device)
original_seq = sample_dec[:, :, :channel_dim]

with torch.no_grad():
    enc_out = base_encoder(sample_enc)
    pred = base_decoder(sample_dec, enc_out, original_seq)

print(f"Prediction shape: {pred.shape}")
print(f"NaNs present: {torch.isnan(pred).any().item()}")
print(f"Sample prediction values (first channel, last Fh steps):")
print(pred[0, -Fh:, 0].cpu().numpy())

In [ ]:
meta_training_data = []
valid_batches = []

# Combine head, body, and tail_train for trajectory generation
all_meta_datasets = head_datasets + body_datasets + tail_train_datasets
np.random.seed(42)  # reproducibility
np.random.shuffle(all_meta_datasets)

print("Generating weight trajectories (full set)...")
for i, ds in enumerate(all_meta_datasets):
    if len(ds) < max(sample_sizes):
        continue
    # No early stopping this time run through all valid datasets

    val_loader_tmp = DataLoader(ds, batch_size=64, shuffle=True)
    val_enc, val_dec, val_tgt = next(iter(val_loader_tmp))
    valid_batches.append((val_enc, val_dec, val_tgt))

    trajectory = []
    for size in sample_sizes:
        theta = get_k_shot_weights(
            base_encoder, base_decoder, ds,
            subset_size=size, epochs_per_shot=EPOCHS_PER_SHOT,
            device=device, channel_dim=channel_dim, Fh=Fh)
        trajectory.append(theta)
    meta_training_data.append(trajectory)
    
    if len(meta_training_data) % 10 == 0:
        print(f"  {len(meta_training_data)} trajectories generated")

print(f"\nDone. Total trajectories: {len(meta_training_data)}")

In [ ]:
torch.save(meta_training_data, 'trajectories_pca.pt')
print(f"Saved {len(meta_training_data)} trajectories to disk")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Flatten all theta vectors across all trajectories
all_thetas = []
all_labels = []
all_sizes = []
for traj_idx in range(len(meta_training_data)):
    for step_idx, theta in enumerate(meta_training_data[traj_idx]):
        all_thetas.append(theta.detach().cpu().numpy().flatten())
        all_labels.append(traj_idx)
        all_sizes.append(sample_sizes[step_idx])
all_thetas = np.array(all_thetas)

# PCA via SVD
mean_theta = all_thetas.mean(axis=0)
centered = all_thetas - mean_theta
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
projected = centered @ Vt[:2].T
var_explained = S[:2]**2 / np.sum(S**2) * 100

fig, ax = plt.subplots(figsize=(7, 5))

# Draw arrows from 50-shot to 800-shot for each trajectory
for traj_idx in range(len(meta_training_data)):
    mask = [i for i, l in enumerate(all_labels) if l == traj_idx]
    pts = projected[mask]
    ax.annotate('',
                xy=pts[-1], xytext=pts[0],
                arrowprops=dict(arrowstyle='->', alpha=0.45,
                                color='grey', lw=1.0))

# Start and end scatter
start_pts = np.array([projected[[i for i, l in enumerate(all_labels)
                                  if l == t][0]]
                      for t in range(len(meta_training_data))])
end_pts = np.array([projected[[i for i, l in enumerate(all_labels)
                                if l == t][-1]]
                    for t in range(len(meta_training_data))])

ax.scatter(start_pts[:, 0], start_pts[:, 1],
           c='#1f77b4', s=50, edgecolors='k', linewidth=0.5,
           label='50-shot (start)', zorder=3)
ax.scatter(end_pts[:, 0], end_pts[:, 1],
           c='#d62728', s=50, edgecolors='k', linewidth=0.5,
           marker='^', label='800-shot (end)', zorder=3)

# Percentile-based axis clipping to exclude extreme outliers
xlim = np.percentile(projected[:, 0], [2, 98])
ylim = np.percentile(projected[:, 1], [2, 98])
pad_x = 0.1 * (xlim[1] - xlim[0])
pad_y = 0.1 * (ylim[1] - ylim[0])
ax.set_xlim(xlim[0] - pad_x, xlim[1] + pad_x)
ax.set_ylim(ylim[0] - pad_y, ylim[1] + pad_y)

ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance)')
ax.legend(fontsize=9, loc='best')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('figure_7_pca.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPC1: {var_explained[0]:.1f}% variance")
print(f"PC2: {var_explained[1]:.1f}% variance")
print(f"PC1 + PC2 total: {var_explained[0] + var_explained[1]:.1f}%")

In [ ]:
meta_training_data = []
valid_batches = []

# Combine head, body, and tail_train for trajectory generation
all_meta_datasets = head_datasets + body_datasets + tail_train_datasets
np.random.shuffle(all_meta_datasets)

print("Generating weight trajectories...")
for i, ds in enumerate(all_meta_datasets):
    if len(ds) < max(sample_sizes):
        continue

    val_loader = DataLoader(ds, batch_size=64, shuffle=True)
    val_enc, val_dec, val_tgt = next(iter(val_loader))
    valid_batches.append((val_enc, val_dec, val_tgt))

    trajectory = []
    for size in sample_sizes:
        theta = get_k_shot_weights(
            base_encoder, base_decoder, ds,
            subset_size=size, epochs_per_shot=EPOCHS_PER_SHOT,
            device=device, channel_dim=channel_dim, Fh=Fh)
        trajectory.append(theta)
    meta_training_data.append(trajectory)

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(all_meta_datasets)} trajectories generated")

print(f"Total trajectories: {len(meta_training_data)}")

param_dim  = 64 * channel_dim + channel_dim   # 1170
num_blocks = len(sample_sizes) - 1            # 1

meta_net = MetaModelNet(param_dim=param_dim, num_blocks=num_blocks).to(device)

In [ ]:
# SPLIT META-TRAINING DATA INTO TRAIN/VAL 
num_meta_total = len(meta_training_data)
num_meta_val   = max(1, int(0.15 * num_meta_total)) # 15% for validation
num_meta_train = num_meta_total - num_meta_val

# They are already shuffled, so we can just slice
meta_train_data    = meta_training_data[:num_meta_train]
meta_train_batches = valid_batches[:num_meta_train]

meta_val_data      = meta_training_data[num_meta_train:]
meta_val_batches   = valid_batches[num_meta_train:]

print(f"Meta-Learning Split: {num_meta_train} train trajectories, {num_meta_val} val trajectories")

# MetaModelNet training
print("Training MetaModelNet")

META_EPOCHS = 500      # Increased because early stopping will catch it
META_PATIENCE = 25     # Stop if val loss doesn't improve for 25 epochs
LR_PATIENCE = 10       # Drop LR if val loss plateaus for 10 epochs

for block_idx in range(num_blocks - 1, -1, -1):
    print(f"\nTraining block {block_idx} "
          f"({sample_sizes[block_idx]}-shot -> {sample_sizes[-1]}-shot)")

    # Train all blocks from block_idx to the end
    params_to_train = []
    for b_idx in range(block_idx, num_blocks):
        params_to_train.extend(list(meta_net.blocks[b_idx].parameters()))

    block_optimizer = torch.optim.Adam(params_to_train, lr=3e-4)
    # Scheduler: Halves the LR if validation loss stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        block_optimizer, mode='min', factor=0.5, patience=LR_PATIENCE, verbose=False)

    best_val_loss = float('inf')
    early_stop_counter = 0
    best_model_state = None

    for epoch in range(1, META_EPOCHS + 1):
        
        # TRAINING PHASE
        meta_net.train()
        train_loss = 0.0

        for traj_idx, trajectory in enumerate(meta_train_data):
            val_enc, val_dec, val_tgt = meta_train_batches[traj_idx]
            val_enc, val_dec, val_tgt = val_enc.to(device), val_dec.to(device), val_tgt.to(device)
            val_original_seq = val_dec[:, :, :channel_dim]
            
            theta_k    = trajectory[block_idx].to(device)
            theta_star = trajectory[-1].to(device)

            block_optimizer.zero_grad()
            theta_pred = meta_net(theta_k, start_block_idx=block_idx)

            # Regression loss
            loss_reg = F.mse_loss(theta_pred, theta_star)

            # Performance loss
            with torch.no_grad():
                enc_out = base_encoder(val_enc)
                features = decoder_forward_features(
                    base_decoder, val_dec, enc_out, val_original_seq)

            w_size = 64 * channel_dim
            w = theta_pred[0, :w_size].view(channel_dim, 64)
            b = theta_pred[0, w_size:]
            pred_perf = features @ w.T + b
            loss_perf = nmse_loss(pred_perf[:, -Fh:, :], val_tgt)

            loss = loss_reg + LAMBDA * loss_perf
            loss.backward()
            block_optimizer.step()
            train_loss += loss.item()

        train_loss /= len(meta_train_data)

        # VALIDATION PHASE
        meta_net.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for traj_idx, trajectory in enumerate(meta_val_data):
                val_enc, val_dec, val_tgt = meta_val_batches[traj_idx]
                val_enc, val_dec, val_tgt = val_enc.to(device), val_dec.to(device), val_tgt.to(device)
                val_original_seq = val_dec[:, :, :channel_dim]
                
                theta_k    = trajectory[block_idx].to(device)
                theta_star = trajectory[-1].to(device)

                theta_pred = meta_net(theta_k, start_block_idx=block_idx)
                loss_reg = F.mse_loss(theta_pred, theta_star)

                enc_out = base_encoder(val_enc)
                features = decoder_forward_features(
                    base_decoder, val_dec, enc_out, val_original_seq)

                w_size = 64 * channel_dim
                w = theta_pred[0, :w_size].view(channel_dim, 64)
                b = theta_pred[0, w_size:]
                pred_perf = features @ w.T + b
                loss_perf = nmse_loss(pred_perf[:, -Fh:, :], val_tgt)

                loss = loss_reg + LAMBDA * loss_perf
                val_loss += loss.item()

        val_loss /= len(meta_val_data)
        
        # Step the scheduler based on validation loss
        scheduler.step(val_loss)

        # EARLY STOPPING CHECK
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stop_counter = 0
            # Save the best weights found so far
            best_model_state = copy.deepcopy(meta_net.state_dict())
        else:
            early_stop_counter += 1

        if epoch % 10 == 0:
            current_lr = block_optimizer.param_groups[0]['lr']
            print(f"  Block {block_idx} | Epoch {epoch:03d}/{META_EPOCHS} | "
                  f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {current_lr:.1e}")

        if early_stop_counter >= META_PATIENCE:
            print(f"  -> Early stopping triggered at epoch {epoch}. Restoring best weights (Val Loss: {best_val_loss:.6f})")
            break

    # Restore the best weights before moving to the next block or evaluation
    if best_model_state is not None:
        meta_net.load_state_dict(best_model_state)

print("\nMetaModelNet training complete.")

# Quick convergence check on the Validation Set:
meta_net.eval()
with torch.no_grad():
    check_loss = 0.0
    for traj_idx, trajectory in enumerate(meta_val_data):
        theta_k = trajectory[0].to(device)      # 50-shot (block 0 input)
        theta_star = trajectory[-1].to(device)   # 800-shot target
        theta_pred = meta_net(theta_k, start_block_idx=0)
        check_loss += F.mse_loss(theta_pred, theta_star).item()
    print(f"Block 0 final pure regression loss (on Validation Set): {check_loss/len(meta_val_data):.6f}")

In [ ]:
# Quick convergence check: run one pass to see final loss
meta_net.eval()
with torch.no_grad():
    check_loss = 0.0
    for traj_idx, trajectory in enumerate(meta_training_data):
        theta_k = trajectory[0].to(device)      # 50-shot (block 0 input)
        theta_star = trajectory[-1].to(device)   # 800-shot target
        theta_pred = meta_net(theta_k, start_block_idx=0)
        check_loss += F.mse_loss(theta_pred, theta_star).item()
    print(f"Block 0 final regression loss: {check_loss/len(meta_training_data):.6f}")

In [ ]:
# Ablation: MetaModelNet performance at different entry points
print("Testing MetaModelNet at different input budgets (no retraining)...\n")

meta_net.eval()

for entry_idx, entry_size in enumerate(sample_sizes[:-1]):
    nmse_bl = []
    nmse_mt = []
    
    for task_idx, tail_ds in enumerate(tail_eval_datasets):
        if len(tail_ds) < 10:
            continue
        
        tail_loader = DataLoader(tail_ds, batch_size=512, shuffle=False)
        
        loss_baseline = evaluate_model(
            base_encoder, base_decoder, tail_loader, device, Fh, channel_dim)
        
        # Generate k-shot weights at this budget
        theta_k = get_k_shot_weights(
            base_encoder, base_decoder, tail_ds,
            subset_size=entry_size, epochs_per_shot=EPOCHS_PER_SHOT,
            device=device, channel_dim=channel_dim, Fh=Fh)
        
        # Transform through remaining blocks
        with torch.no_grad():
            theta_meta = meta_net(theta_k.to(device), start_block_idx=entry_idx)
        
        meta_decoder = copy.deepcopy(base_decoder).to(device)
        meta_decoder = inject_weights(
            meta_decoder, theta_meta, in_features=64, channel_dim=channel_dim)
        loss_meta = evaluate_model(
            base_encoder, meta_decoder, tail_loader, device, Fh, channel_dim)
        
        nmse_bl.append(loss_baseline)
        nmse_mt.append(loss_meta)
    
    mean_bl = 10*np.log10(np.mean(nmse_bl))
    mean_mt = 10*np.log10(np.mean(nmse_mt))
    wins = sum(1 for m, b in zip(nmse_mt, nmse_bl) if m < b)
    
    print(f"{entry_size:>4d}-shot (blocks {entry_idx}->{num_blocks-1}) | "
          f"Baseline: {mean_bl:.2f} dB | MetaModel: {mean_mt:.2f} dB | "
          f"Gain: {mean_mt - mean_bl:.2f} dB | Wins: {wins}/{len(nmse_bl)}")

print(f"\n(Baseline is the same across all rows: unadapted T-LSTM)")

In [ ]:
print("\n Evaluating on tail (hard) tasks \n")

nmse_baseline_list = []
nmse_meta_list     = []
nmse_finetune_list = []

meta_net.eval()

for task_idx, tail_ds in enumerate(tail_eval_datasets):
    if len(tail_ds) < 10:
        continue

    tail_loader = DataLoader(tail_ds, batch_size=512, shuffle=False)

    # 1. T-LSTM alone (no adaptation)
    loss_baseline = evaluate_model(
        base_encoder, base_decoder, tail_loader, device, Fh, channel_dim)

    # Split few-shot budget: 50 for MetaModelNet input, separate 50 for fine-tuning
    all_indices = np.random.permutation(len(tail_ds))
    meta_indices = all_indices[:50]
    ft_indices   = all_indices[50:100]

    # 2. T-LSTM + 50-shot fine-tuning (uses ft_indices only)
    ft_subset_ds = Subset(tail_ds, ft_indices)
    enc_ft = copy.deepcopy(base_encoder).to(device)
    dec_ft = copy.deepcopy(base_decoder).to(device)
    for param in dec_ft.parameters():
        param.requires_grad = False
    for param in dec_ft.fc[3].parameters():
        param.requires_grad = True

    ft_loader = DataLoader(ft_subset_ds, batch_size=32, shuffle=True)
    opt_ft = torch.optim.Adam(dec_ft.fc[3].parameters(), lr=5e-3)
    enc_ft.eval()
    dec_ft.eval()

    for _ in range(EPOCHS_PER_SHOT):
        for enc_in, dec_in, target in ft_loader:
            enc_in = enc_in.to(device)
            dec_in = dec_in.to(device)
            target = target.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            opt_ft.zero_grad()
            with torch.no_grad():
                encoded = enc_ft(enc_in)
            pred = dec_ft(dec_in, encoded, original_seq)
            loss = nmse_loss(pred[:, -Fh:, :], target)
            loss.backward()
            opt_ft.step()

    theta_finetuned = extract_weights(dec_ft)
    ft_decoder = copy.deepcopy(base_decoder).to(device)
    ft_decoder = inject_weights(
        ft_decoder, theta_finetuned, in_features=64, channel_dim=channel_dim)
    loss_finetune = evaluate_model(
        base_encoder, ft_decoder, tail_loader, device, Fh, channel_dim)

    # 3. T-LSTM + MetaModelNet (uses meta_indices only)
    meta_subset = Subset(tail_ds, meta_indices)
    meta_loader = DataLoader(meta_subset, batch_size=32, shuffle=True)

    enc_clone = copy.deepcopy(base_encoder).to(device)
    dec_clone = copy.deepcopy(base_decoder).to(device)
    for param in dec_clone.parameters():
        param.requires_grad = False
    for param in dec_clone.fc[3].parameters():
        param.requires_grad = True

    opt = torch.optim.Adam(dec_clone.fc[3].parameters(), lr=5e-3)
    enc_clone.eval()
    dec_clone.eval()

    for _ in range(EPOCHS_PER_SHOT):
        for enc_in, dec_in, target in meta_loader:
            enc_in = enc_in.to(device)
            dec_in = dec_in.to(device)
            target = target.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            opt.zero_grad()
            with torch.no_grad():
                encoded = enc_clone(enc_in)
            pred = dec_clone(dec_in, encoded, original_seq)
            loss = nmse_loss(pred[:, -Fh:, :], target)
            loss.backward()
            opt.step()

    theta_for_meta = extract_weights(dec_clone)
    with torch.no_grad():
        theta_meta = meta_net(theta_for_meta.to(device), start_block_idx=0)

    meta_decoder = copy.deepcopy(base_decoder).to(device)
    meta_decoder = inject_weights(
        meta_decoder, theta_meta, in_features=64, channel_dim=channel_dim)
    loss_meta = evaluate_model(
        base_encoder, meta_decoder, tail_loader, device, Fh, channel_dim)

    nmse_baseline_list.append(loss_baseline)
    nmse_finetune_list.append(loss_finetune)
    nmse_meta_list.append(loss_meta)

    print(f"Tail {task_idx+1:02d} | "
          f"T-LSTM: {10*np.log10(loss_baseline):.2f} dB | "
          f"50-shot: {10*np.log10(loss_finetune):.2f} dB | "
          f"MetaModel: {10*np.log10(loss_meta):.2f} dB")

# --- Summary ---
print(f"\n--- Mean across {len(nmse_baseline_list)} held-out tail tasks ---")
print(f"T-LSTM (no adapt):     {10*np.log10(np.mean(nmse_baseline_list)):.2f} dB")
print(f"T-LSTM + 50-shot FT:   {10*np.log10(np.mean(nmse_finetune_list)):.2f} dB")
print(f"T-LSTM + MetaModelNet: {10*np.log10(np.mean(nmse_meta_list)):.2f} dB")

meta_wins = sum(1 for m, b in zip(nmse_meta_list, nmse_baseline_list) if m < b)
n_tasks = len(nmse_baseline_list)
print(f"\nMetaModelNet beats baseline on {meta_wins}/{n_tasks} tasks")

In [ ]:
meta_wins = sum(1 for m, b in zip(nmse_meta_list, nmse_baseline_list) if m < b)
n_tasks = len(nmse_baseline_list)
print(f"MetaModelNet beats baseline on {meta_wins}/{n_tasks} tasks")

In [ ]:
# Visualisation of results
import matplotlib.pyplot as plt

results_dB = {
    'T-LSTM':       [10*np.log10(x) for x in nmse_baseline_list],
    'MetaModelNet': [10*np.log10(x) for x in nmse_meta_list],
    '50-shot FT':   [10*np.log10(x) for x in nmse_finetune_list],
}
n_tasks = len(nmse_baseline_list)
task_labels = [f'T{i+1:02d}' for i in range(n_tasks)]

colours = {
    'T-LSTM':       '#888780',
    'MetaModelNet': '#7F77DD',
    '50-shot FT':   '#1D9E75',
}

#  Figure 1: NMSE per task (grouped bar chart, sorted by baseline difficulty)
sort_idx = np.argsort(results_dB['T-LSTM'])
methods = list(results_dB.keys())
x = np.arange(n_tasks)
bar_w = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
for j, method in enumerate(methods):
    vals = [results_dB[method][i] for i in sort_idx]
    ax.bar(x + j * bar_w, vals, bar_w, label=method, color=colours[method])

ax.set_xticks(x + bar_w)
ax.set_xticklabels([task_labels[i] for i in sort_idx], rotation=45, fontsize=8)
ax.set_ylabel('NMSE (dB)')
ax.set_title('NMSE per held-out tail task (sorted by baseline difficulty)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Figure 2: MetaModelNet gain per task
meta_gain = [results_dB['MetaModelNet'][i] - results_dB['T-LSTM'][i] for i in range(n_tasks)]

fig, ax = plt.subplots(figsize=(10, 4))
bar_col = ['#7F77DD' if v < 0 else '#E24B4A' for v in meta_gain]
ax.bar(task_labels, meta_gain, color=bar_col)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Gain (dB)')
ax.set_title('MetaModelNet gain over T-LSTM baseline (negative = improvement)')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Figure 3: CDF of NMSE
label_map_cdf = {
    'T-LSTM':       'Baseline (no adaptation)',
    'MetaModelNet': 'MetaModelNet (this work)',
    '50-shot FT':   'Supervised retraining',
}

fig, ax = plt.subplots(figsize=(8, 5))
for method in methods:
    sorted_vals = np.sort(results_dB[method])
    cdf = np.arange(1, n_tasks + 1) / n_tasks
    ax.step(sorted_vals, cdf, where='post',
            label=label_map_cdf[method],
            color=colours[method], linewidth=2.5)

ax.set_xlabel('Prediction Error — lower is better (dB)', fontsize=13)
ax.set_ylabel('Proportion of conditions', fontsize=13)
ax.set_title('Distribution of prediction accuracy\nacross all 24 unseen rare conditions',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.tick_params(labelsize=11)
ax.grid(alpha=0.3)

# Annotation pointing out the leftward shift
ax.annotate(
    'MetaModelNet consistently\nbetter than baseline',
    xy=(-22, 0.5),
    xytext=(-15, 0.25),
    fontsize=10, color='#7F77DD', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#7F77DD', lw=1.5)
)

plt.tight_layout()
plt.savefig('cdf_poster.png', dpi=300, bbox_inches='tight')
plt.show()

#  Figure 4: Mean NMSE summary bar chart
label_map = {
    'T-LSTM':       'Baseline\n(no adaptation)',
    'MetaModelNet': 'MetaModelNet\n(this work)',
    '50-shot FT':   'Supervised\nretraining',
}
means = {m: 10*np.log10(np.mean(lst)) for m, lst in zip(methods,
         [nmse_baseline_list, nmse_meta_list, nmse_finetune_list])}

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    [label_map[m] for m in means.keys()],
    means.values(),
    color=[colours[m] for m in means.keys()],
    width=0.5
)
ax.set_ylabel('Mean Prediction Error — lower is better (dB)', fontsize=13)
ax.set_title('Average prediction accuracy across\n24 unseen rare conditions', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=11)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, means.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.3,
            f'{val:.2f} dB', ha='center', va='top',
            fontsize=12, fontweight='bold', color='white')

# Annotation showing the improvement
ax.annotate(
    '6.51 dB improvement',
    xy=(1, means['MetaModelNet']),
    xytext=(1.6, means['MetaModelNet'] + 4),
    fontsize=11, color='#7F77DD', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#7F77DD', lw=1.5)
)

plt.tight_layout()
plt.savefig('mean_nmse_poster.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction horizon sweep (Fh = 2 to 7)

horizon_results = {}

for fh_eval in range(2, 8):
    print(f"\n{'='*60}")
    print(f"Evaluating at Fh = {fh_eval}")
    print(f"{'='*60}")
    
    eval_datasets_fh = [
        TimeDataset(chunk, P, G, fh_eval, A) 
        for chunk in tail_eval_chunks
    ]
    
    nmse_bl = []
    nmse_mt = []
    nmse_ft = []
    
    meta_net.eval()
    
    for task_idx, tail_ds in enumerate(eval_datasets_fh):
        if len(tail_ds) < 10:
            continue
        
        tail_loader = DataLoader(tail_ds, batch_size=512, shuffle=False)
        
        # 1. T-LSTM alone
        loss_baseline = evaluate_model(
            base_encoder, base_decoder, tail_loader, device, fh_eval, channel_dim)
        
        all_indices = np.random.permutation(len(tail_ds))
        meta_indices = all_indices[:50]
        ft_indices   = all_indices[50:100]
        
        # 2. 50-shot fine-tuning
        ft_subset_ds = Subset(tail_ds, ft_indices)
        enc_ft = copy.deepcopy(base_encoder).to(device)
        dec_ft = copy.deepcopy(base_decoder).to(device)
        for param in dec_ft.parameters():
            param.requires_grad = False
        for param in dec_ft.fc[3].parameters():
            param.requires_grad = True
        
        ft_loader = DataLoader(ft_subset_ds, batch_size=32, shuffle=True)
        opt_ft = torch.optim.Adam(dec_ft.fc[3].parameters(), lr=5e-3)
        enc_ft.eval()
        dec_ft.eval()
        
        for _ in range(EPOCHS_PER_SHOT):
            for enc_in, dec_in, target in ft_loader:
                enc_in = enc_in.to(device)
                dec_in = dec_in.to(device)
                target = target.to(device)
                original_seq = dec_in[:, :, :channel_dim]
                opt_ft.zero_grad()
                with torch.no_grad():
                    encoded = enc_ft(enc_in)
                pred = dec_ft(dec_in, encoded, original_seq)
                loss = nmse_loss(pred[:, -fh_eval:, :], target)
                loss.backward()
                opt_ft.step()
        
        theta_finetuned = extract_weights(dec_ft)
        ft_decoder = copy.deepcopy(base_decoder).to(device)
        ft_decoder = inject_weights(ft_decoder, theta_finetuned, in_features=64, channel_dim=channel_dim)
        loss_finetune = evaluate_model(
            base_encoder, ft_decoder, tail_loader, device, fh_eval, channel_dim)
        
        # 3. MetaModelNet
        meta_subset = Subset(tail_ds, meta_indices)
        meta_loader = DataLoader(meta_subset, batch_size=32, shuffle=True)
        
        enc_clone = copy.deepcopy(base_encoder).to(device)
        dec_clone = copy.deepcopy(base_decoder).to(device)
        for param in dec_clone.parameters():
            param.requires_grad = False
        for param in dec_clone.fc[3].parameters():
            param.requires_grad = True
        
        opt = torch.optim.Adam(dec_clone.fc[3].parameters(), lr=5e-3)
        enc_clone.eval()
        dec_clone.eval()
        
        for _ in range(EPOCHS_PER_SHOT):
            for enc_in, dec_in, target in meta_loader:
                enc_in = enc_in.to(device)
                dec_in = dec_in.to(device)
                target = target.to(device)
                original_seq = dec_in[:, :, :channel_dim]
                opt.zero_grad()
                with torch.no_grad():
                    encoded = enc_clone(enc_in)
                pred = dec_clone(dec_in, encoded, original_seq)
                loss = nmse_loss(pred[:, -fh_eval:, :], target)
                loss.backward()
                opt.step()
        
        theta_for_meta = extract_weights(dec_clone)
        with torch.no_grad():
            theta_meta = meta_net(theta_for_meta.to(device), start_block_idx=0)
        
        meta_decoder = copy.deepcopy(base_decoder).to(device)
        meta_decoder = inject_weights(meta_decoder, theta_meta, in_features=64, channel_dim=channel_dim)
        loss_meta = evaluate_model(
            base_encoder, meta_decoder, tail_loader, device, fh_eval, channel_dim)
        
        nmse_bl.append(loss_baseline)
        nmse_mt.append(loss_meta)
        nmse_ft.append(loss_finetune)
    
    horizon_results[fh_eval] = {
        'baseline': 10*np.log10(np.mean(nmse_bl)),
        'meta':     10*np.log10(np.mean(nmse_mt)),
        'finetune': 10*np.log10(np.mean(nmse_ft)),
        'meta_wins': sum(1 for m, b in zip(nmse_mt, nmse_bl) if m < b),
        'n_tasks': len(nmse_bl),
    }
    
    r = horizon_results[fh_eval]
    print(f"  T-LSTM:      {r['baseline']:.2f} dB")
    print(f"  MetaModelNet:{r['meta']:.2f} dB  (wins {r['meta_wins']}/{r['n_tasks']})")
    print(f"  50-shot FT:  {r['finetune']:.2f} dB")

# Plot
fh_vals = sorted(horizon_results.keys())

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fh_vals, [horizon_results[f]['baseline'] for f in fh_vals],
        'o-', color='#888780', linewidth=2, markersize=7, label='T-LSTM (no adapt)')
ax.plot(fh_vals, [horizon_results[f]['meta'] for f in fh_vals],
        's-', color='#7F77DD', linewidth=2, markersize=7, label='T-LSTM + MetaModelNet')
ax.plot(fh_vals, [horizon_results[f]['finetune'] for f in fh_vals],
        '^-', color='#1D9E75', linewidth=2, markersize=7, label='T-LSTM + 50-shot FT')

ax.set_xlabel('Prediction horizon F (steps)', fontsize=12)
ax.set_ylabel('Mean NMSE (dB)', fontsize=12)
ax.set_title('NMSE vs prediction horizon across held-out tail tasks', fontsize=13)
ax.set_xticks(fh_vals)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Summary table
print("\n--- NMSE vs Prediction Horizon ---")
print(f"{'Fh':>4s} | {'T-LSTM':>10s} | {'MetaModel':>10s} | {'50-shot':>10s} | {'Meta wins':>10s}")
print("-" * 60)
for f in fh_vals:
    r = horizon_results[f]
    print(f"{f:4d} | {r['baseline']:>10.2f} | {r['meta']:>10.2f} | {r['finetune']:>10.2f} | {r['meta_wins']:>5d}/{r['n_tasks']:<4d}")

In [ ]:
# Task metadata analysis
import h5py

with h5py.File(MAT_PATH, 'r') as f:
    tail_cell = f['Dataset_Tail']
    refs = tail_cell[()].flatten()
    
    all_tail_meta = []
    for ref in refs:
        grp = f[ref]
        meta = {
            'Elevation': float(grp['Elevation'][()].flatten()[0]),
            'Speed': float(grp['Speed'][()].flatten()[0]),
            'fc': float(grp['fc'][()].flatten()[0]),
            'Difficulty': float(grp['Difficulty'][()].flatten()[0]),
        }
        # Environment is stored as uint16 array, need to decode
        try:
            env_ref = grp['Environment'][()].flatten()
            env_chars = ''.join([chr(c) for c in f[env_ref[0]][()].flatten()])
            meta['Environment'] = env_chars
        except:
            meta['Environment'] = 'Unknown'
        all_tail_meta.append(meta)

# Map to eval tasks using the same seed-42 split
np.random.seed(42)
tail_indices = np.random.permutation(len(all_tail_meta))
eval_idx = tail_indices[30:]

print(f"{'Task':>6s} | {'Elev':>6s} | {'Speed':>6s} | {'fc GHz':>7s} | {'Score':>6s} | {'T-LSTM':>8s} | {'Meta':>8s} | {'Gain':>8s} | {'Result':>8s}")
print("-" * 95)

results_bl = [10*np.log10(x) for x in nmse_baseline_list]
results_mt = [10*np.log10(x) for x in nmse_meta_list]

for i, idx in enumerate(eval_idx):
    m = all_tail_meta[idx]
    gain = results_mt[i] - results_bl[i]
    result = "WIN" if gain < 0 else "LOSS"
    print(f"T{i+1:02d}    | {m['Elevation']:6.1f} | {m['Speed']:6.0f} | {m['fc']/1e9:7.0f} | {m['Difficulty']:6.2f} | {results_bl[i]:8.2f} | {results_mt[i]:8.2f} | {gain:8.2f} | {result:>8s}")

In [ ]:
# Weight space visualisation (PCA)

all_thetas = []
all_labels = []
all_sizes  = []

for traj_idx in range(min(20, len(meta_training_data))):
    for step_idx, theta in enumerate(meta_training_data[traj_idx]):
        all_thetas.append(theta.detach().cpu().numpy().flatten())
        all_labels.append(traj_idx)
        all_sizes.append(sample_sizes[step_idx])

all_thetas = np.array(all_thetas)

# Manual PCA using numpy SVD
mean_theta = all_thetas.mean(axis=0)
centered = all_thetas - mean_theta
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
projected = centered @ Vt[:2].T
var_explained = S[:2]**2 / np.sum(S**2) * 100

fig, ax = plt.subplots(figsize=(8, 6))

cmap = plt.cm.coolwarm
for traj_idx in range(min(20, len(meta_training_data))):
    mask = [i for i, l in enumerate(all_labels) if l == traj_idx]
    pts = projected[mask]
    sizes_traj = [all_sizes[i] for i in mask]
    
    ax.plot(pts[:, 0], pts[:, 1], 'k-', alpha=0.15, linewidth=0.8)
    
    for j, (pt, sz) in enumerate(zip(pts, sizes_traj)):
        colour = cmap(j / (len(sample_sizes) - 1))
        ax.scatter(pt[0], pt[1], c=[colour], s=40, edgecolors='k', linewidth=0.3, zorder=3)

for j, sz in enumerate(sample_sizes):
    colour = cmap(j / (len(sample_sizes) - 1))
    ax.scatter([], [], c=[colour], s=60, edgecolors='k', linewidth=0.3, label=f'{sz}-shot')

ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance)')
ax.set_title('Weight trajectories in parameter space (PCA)')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Multi-seed check
print("Running multi-seed robustness check\n")

seed_results = {}

for seed in [0, 1, 2, 3, 4]:
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    nmse_bl_seed = []
    nmse_mt_seed = []
    nmse_ft_seed = []
    
    meta_net.eval()
    
    for task_idx, tail_ds in enumerate(tail_eval_datasets):
        if len(tail_ds) < 10:
            continue
        
        tail_loader = DataLoader(tail_ds, batch_size=512, shuffle=False)
        
        # Baseline (deterministic, no randomness)
        loss_baseline = evaluate_model(
            base_encoder, base_decoder, tail_loader, device, Fh, channel_dim)
        
        # Split indices with this seed
        all_indices = np.random.permutation(len(tail_ds))
        meta_indices = all_indices[:50]
        ft_indices   = all_indices[50:100]
        
        # 50-shot fine-tuning
        ft_subset_ds = Subset(tail_ds, ft_indices)
        enc_ft = copy.deepcopy(base_encoder).to(device)
        dec_ft = copy.deepcopy(base_decoder).to(device)
        for param in dec_ft.parameters():
            param.requires_grad = False
        for param in dec_ft.fc[3].parameters():
            param.requires_grad = True
        
        ft_loader = DataLoader(ft_subset_ds, batch_size=32, shuffle=True)
        opt_ft = torch.optim.Adam(dec_ft.fc[3].parameters(), lr=5e-3, weight_decay=1e-4)
        enc_ft.eval()
        dec_ft.eval()
        
        for _ in range(EPOCHS_PER_SHOT):
            for enc_in, dec_in, target in ft_loader:
                enc_in = enc_in.to(device)
                dec_in = dec_in.to(device)
                target = target.to(device)
                original_seq = dec_in[:, :, :channel_dim]
                opt_ft.zero_grad()
                with torch.no_grad():
                    encoded = enc_ft(enc_in)
                pred = dec_ft(dec_in, encoded, original_seq)
                loss = nmse_loss(pred[:, -Fh:, :], target)
                loss.backward()
                opt_ft.step()
        
        theta_finetuned = extract_weights(dec_ft)
        ft_decoder = copy.deepcopy(base_decoder).to(device)
        ft_decoder = inject_weights(ft_decoder, theta_finetuned, in_features=64, channel_dim=channel_dim)
        loss_finetune = evaluate_model(
            base_encoder, ft_decoder, tail_loader, device, Fh, channel_dim)
        
        # MetaModelNet
        meta_subset = Subset(tail_ds, meta_indices)
        meta_loader = DataLoader(meta_subset, batch_size=32, shuffle=True)
        
        enc_clone = copy.deepcopy(base_encoder).to(device)
        dec_clone = copy.deepcopy(base_decoder).to(device)
        for param in dec_clone.parameters():
            param.requires_grad = False
        for param in dec_clone.fc[3].parameters():
            param.requires_grad = True
        
        opt = torch.optim.Adam(dec_clone.fc[3].parameters(), lr=5e-3, weight_decay=1e-4)
        enc_clone.eval()
        dec_clone.eval()
        
        for _ in range(EPOCHS_PER_SHOT):
            for enc_in, dec_in, target in meta_loader:
                enc_in = enc_in.to(device)
                dec_in = dec_in.to(device)
                target = target.to(device)
                original_seq = dec_in[:, :, :channel_dim]
                opt.zero_grad()
                with torch.no_grad():
                    encoded = enc_clone(enc_in)
                pred = dec_clone(dec_in, encoded, original_seq)
                loss = nmse_loss(pred[:, -Fh:, :], target)
                loss.backward()
                opt.step()
        
        theta_for_meta = extract_weights(dec_clone)
        with torch.no_grad():
            theta_meta = meta_net(theta_for_meta.to(device), start_block_idx=0)
        
        meta_decoder = copy.deepcopy(base_decoder).to(device)
        meta_decoder = inject_weights(meta_decoder, theta_meta, in_features=64, channel_dim=channel_dim)
        loss_meta = evaluate_model(
            base_encoder, meta_decoder, tail_loader, device, Fh, channel_dim)
        
        nmse_bl_seed.append(loss_baseline)
        nmse_mt_seed.append(loss_meta)
        nmse_ft_seed.append(loss_finetune)
    
    wins = sum(1 for m, b in zip(nmse_mt_seed, nmse_bl_seed) if m < b)
    seed_results[seed] = {
        'baseline': 10*np.log10(np.mean(nmse_bl_seed)),
        'meta':     10*np.log10(np.mean(nmse_mt_seed)),
        'finetune': 10*np.log10(np.mean(nmse_ft_seed)),
        'wins':     wins,
        'n_tasks':  len(nmse_bl_seed),
    }
    
    r = seed_results[seed]
    print(f"Seed {seed} | T-LSTM: {r['baseline']:.2f} dB | "
          f"MetaModel: {r['meta']:.2f} dB | "
          f"50-shot: {r['finetune']:.2f} dB | "
          f"Wins: {r['wins']}/{r['n_tasks']}")

# Summary statistics across seeds
all_bl = [seed_results[s]['baseline'] for s in seed_results]
all_mt = [seed_results[s]['meta'] for s in seed_results]
all_ft = [seed_results[s]['finetune'] for s in seed_results]
all_wins = [seed_results[s]['wins'] for s in seed_results]

print(f"\n--- Across 5 seeds ---")
print(f"T-LSTM:      {np.mean(all_bl):.2f} +/- {np.std(all_bl):.2f} dB")
print(f"MetaModelNet:{np.mean(all_mt):.2f} +/- {np.std(all_mt):.2f} dB")
print(f"50-shot FT:  {np.mean(all_ft):.2f} +/- {np.std(all_ft):.2f} dB")
print(f"Win rate:    {np.mean(all_wins):.1f} +/- {np.std(all_wins):.1f} out of 24")

In [ ]:
import matplotlib.pyplot as plt

# Pick a tail task where MetaModelNet shows clear improvement
# Use a fixed task and sample for reproducibility
eval_ds = tail_eval_datasets[2]  # Task 03 had 11.98 dB gain
eval_loader_viz = DataLoader(eval_ds, batch_size=64, shuffle=False)
eval_enc, eval_dec, eval_tgt = next(iter(eval_loader_viz))
eval_enc = eval_enc.to(device)
eval_dec = eval_dec.to(device)
eval_tgt = eval_tgt.to(device)

base_encoder.eval()
base_decoder.eval()
meta_decoder.eval()

with torch.no_grad():
    enc_out = base_encoder(eval_enc)
    base_pred = base_decoder(eval_dec, enc_out, eval_dec[:, :, :channel_dim])
    meta_pred = meta_decoder(eval_dec, enc_out, eval_dec[:, :, :channel_dim])

# Plot multiple samples to show consistency
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Channel prediction examples (Tail task 03, Re{H} antenna 1)', fontsize=13)

for idx, ax in enumerate(axes.flat):
    sample = idx * 10  # spread across different time positions
    
    gt = eval_tgt[sample, :, 0].cpu().numpy()
    bp = base_pred[sample, -Fh:, 0].cpu().numpy()
    mp = meta_pred[sample, -Fh:, 0].cpu().numpy()
    
    # Include the last few historical points for context
    hist = eval_dec[sample, -Fh-5:-Fh, 0].cpu().numpy()
    
    t_hist = range(-5, 0)
    t_pred = range(0, Fh)
    
    ax.plot(t_hist, hist, 'k-', linewidth=1.5, alpha=0.4, label='History')
    ax.plot(t_pred, gt, 'k-', linewidth=2, label='Ground truth')
    ax.plot(t_pred, bp, '--', color='#888780', linewidth=1.5, label='T-LSTM')
    ax.plot(t_pred, mp, 'o-', color='#7F77DD', linewidth=1.5, markersize=5, label='MetaModelNet')
    
    ax.axvline(0, color='black', linewidth=0.5, alpha=0.3)
    ax.set_xlabel('Step', fontsize=9)
    ax.set_ylabel('Re{H}', fontsize=9)
    ax.set_title(f'Sample {sample}', fontsize=10)
    ax.grid(alpha=0.2)
    
    if idx == 0:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# BER helpers
from scipy.special import erfc

def reconstruct_complex_channel(tensor_18d, mean, std):
    data = tensor_18d * (std + 1e-8) + mean
    return data[:, :9] + 1j * data[:, 9:]

def compute_ber_for_channels(h_true_all, h_pred_all, snr_db_range):
    def _unit(H):
        n = np.linalg.norm(H, axis=1, keepdims=True)
        return H / np.maximum(n, 1e-30)
    ht, hp = _unit(h_true_all), _unit(h_pred_all)
    rho = np.abs(np.sum(np.conj(ht) * hp, axis=1)) ** 2
    ber = np.zeros(len(snr_db_range))
    for k, snr_db in enumerate(snr_db_range):
        snr_lin = 10.0 ** (snr_db / 10.0)
        ber[k] = np.mean(0.5 * erfc(np.sqrt(rho * snr_lin / 2.0)))
    return ber

def compute_rate(h_true_all, h_pred_all, snr_db_range):
    def _unit(H):
        n = np.linalg.norm(H, axis=1, keepdims=True)
        return H / np.maximum(n, 1e-30)
    ht, hp = _unit(h_true_all), _unit(h_pred_all)
    rho = np.abs(np.sum(np.conj(ht) * hp, axis=1)) ** 2
    rate = np.zeros(len(snr_db_range))
    for k, snr_db in enumerate(snr_db_range):
        snr_lin = 10.0 ** (snr_db / 10.0)
        rate[k] = np.mean(np.log2(1.0 + rho * snr_lin))
    return rate

print("BER + rate helpers defined")


In [ ]:
# Collect channel predictions across tail tasks, reconstruct per task
np.random.seed(0)
torch.manual_seed(0)

meta_net.eval(); base_encoder.eval(); base_decoder.eval()

ch_true, ch_base, ch_mmn, ch_ft = [], [], [], []

for task_idx, tail_ds in enumerate(tail_eval_datasets):
    if len(tail_ds) < 10:
        continue

    tail_loader = DataLoader(tail_ds, batch_size=512, shuffle=False)

    # true + unadapted baseline
    t_true, t_base = [], []
    with torch.no_grad():
        for enc_in, dec_in, target in tail_loader:
            enc_in, dec_in = enc_in.to(device), dec_in.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            pred_bl = base_decoder(dec_in, base_encoder(enc_in), original_seq)
            t_true.append(target.cpu().numpy().reshape(-1, channel_dim))
            t_base.append(pred_bl[:, -Fh:, :].cpu().numpy().reshape(-1, channel_dim))
    true_np = np.concatenate(t_true, axis=0)
    base_np = np.concatenate(t_base, axis=0)

    # disjoint 50/50 budgets (report protocol)
    perm = np.random.permutation(len(tail_ds))
    meta_idx, ft_idx = perm[:50], perm[50:100]

    # 50-shot supervised fine-tune (wd=1e-4, matches trajectory generation)
    dec_ft = copy.deepcopy(base_decoder).to(device)
    for p in dec_ft.parameters():           p.requires_grad = False
    for p in dec_ft.fc[3].parameters():     p.requires_grad = True
    ft_loader = DataLoader(Subset(tail_ds, ft_idx), batch_size=32, shuffle=True)
    opt_ft = torch.optim.Adam(dec_ft.fc[3].parameters(), lr=5e-3, weight_decay=1e-4)
    dec_ft.eval()
    for _ in range(EPOCHS_PER_SHOT):
        for enc_in, dec_in, target in ft_loader:
            enc_in, dec_in, target = enc_in.to(device), dec_in.to(device), target.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            opt_ft.zero_grad()
            with torch.no_grad(): encoded = base_encoder(enc_in)
            loss = nmse_loss(dec_ft(dec_in, encoded, original_seq)[:, -Fh:, :], target)
            loss.backward(); opt_ft.step()
    ft_dec = inject_weights(copy.deepcopy(base_decoder).to(device),
                            extract_weights(dec_ft), in_features=64, channel_dim=channel_dim)

    # MMN: 50-shot theta then single forward pass (wd=1e-4 keeps theta in-distribution)
    dec_k = copy.deepcopy(base_decoder).to(device)
    for p in dec_k.parameters():            p.requires_grad = False
    for p in dec_k.fc[3].parameters():      p.requires_grad = True
    k_loader = DataLoader(Subset(tail_ds, meta_idx), batch_size=32, shuffle=True)
    opt_k = torch.optim.Adam(dec_k.fc[3].parameters(), lr=5e-3, weight_decay=1e-4)
    dec_k.eval()
    for _ in range(EPOCHS_PER_SHOT):
        for enc_in, dec_in, target in k_loader:
            enc_in, dec_in, target = enc_in.to(device), dec_in.to(device), target.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            opt_k.zero_grad()
            with torch.no_grad(): encoded = base_encoder(enc_in)
            loss = nmse_loss(dec_k(dec_in, encoded, original_seq)[:, -Fh:, :], target)
            loss.backward(); opt_k.step()
    with torch.no_grad():
        theta_mmn = meta_net(extract_weights(dec_k).to(device), start_block_idx=0)
    mmn_dec = inject_weights(copy.deepcopy(base_decoder).to(device),
                             theta_mmn, in_features=64, channel_dim=channel_dim)

    # collect FT + MMN predictions
    t_ft, t_mmn = [], []
    with torch.no_grad():
        for enc_in, dec_in, target in tail_loader:
            enc_in, dec_in = enc_in.to(device), dec_in.to(device)
            original_seq = dec_in[:, :, :channel_dim]
            enc_out = base_encoder(enc_in)
            t_ft.append(ft_dec(dec_in, enc_out, original_seq)[:, -Fh:, :].cpu().numpy().reshape(-1, channel_dim))
            t_mmn.append(mmn_dec(dec_in, enc_out, original_seq)[:, -Fh:, :].cpu().numpy().reshape(-1, channel_dim))
    ft_np  = np.concatenate(t_ft, axis=0)
    mmn_np = np.concatenate(t_mmn, axis=0)

    # reconstruct with THIS TASK's stats (the critical step)
    m, s = tail_ds.mean, tail_ds.std
    ch_true.append(reconstruct_complex_channel(true_np, m, s))
    ch_base.append(reconstruct_complex_channel(base_np, m, s))
    ch_ft.append(  reconstruct_complex_channel(ft_np,   m, s))
    ch_mmn.append( reconstruct_complex_channel(mmn_np,  m, s))
    print(f"  tail {task_idx+1:02d} done ({len(true_np)} realisations)")

h_true = np.concatenate(ch_true, axis=0)
h_base = np.concatenate(ch_base, axis=0)
h_ft   = np.concatenate(ch_ft,   axis=0)
h_mmn  = np.concatenate(ch_mmn,  axis=0)
print(f"\nTotal realisations: {len(h_true)}  (pooled across tasks)")

In [ ]:
import numpy as np
from scipy.special import erfc

# Monte Carlo function
def ber_monte_carlo(h_true_all, h_pred_all, snr_db_range,
                    n_sym=1000, batch_size=500, seed=0):
    
    rng = np.random.default_rng(seed)

    def _unit(H):
        n = np.linalg.norm(H, axis=1, keepdims=True)
        return H / np.maximum(n, 1e-30)

    ht = _unit(h_true_all)          # (N, Na)
    hp = _unit(h_pred_all)          # (N, Na)
    N  = ht.shape[0]

    # Effective scalar gain per realisation:  g_i = h_true_i^H  h_pred_i
    # (hp is already unit-norm, so w_i = hp_i)
    g = np.sum(np.conj(ht) * hp, axis=1)   # (N,) complex

    ber = np.zeros(len(snr_db_range))

    for k, snr_db in enumerate(snr_db_range):
        snr_lin = 10.0 ** (snr_db / 10.0)
        # sigma is per real dimension; Es = 1 (unit-power QPSK).
        sigma = np.sqrt(1.0 / (2.0 * snr_lin))

        total_errs = 0
        total_bits = 0

        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            B   = end - start
            g_b = g[start:end]                          # (B,)

            # transmit 
            bits = rng.integers(0, 2, size=(B, n_sym, 2))
            #  b=0 -> +1,  b=1 -> -1   (Gray QPSK, unit average power)
            sym = ((1 - 2*bits[:, :, 0])
                   + 1j*(1 - 2*bits[:, :, 1])) / np.sqrt(2)   # (B, n_sym)

            # channel + noise 
            y = g_b[:, None] * sym                      # noiseless Rx
            noise = sigma * (rng.standard_normal((B, n_sym))
                             + 1j * rng.standard_normal((B, n_sym)))
            y = y + noise

            # coherent receiver: remove the phase of g 
            eq = np.conj(g_b) / (np.abs(g_b) + 1e-30)  # (B,)
            y  = y * eq[:, None]
            # After equalisation:  y ≈ |g| s + n'
            # with n' having the same variance (|eq|=1).

            # hard decision (matches the modulation mapping) 
            # real(y) > 0  ->  bit 0   (was mapped to +1)
            # real(y) < 0  ->  bit 1   (was mapped to -1)
            b0_hat = (y.real < 0).astype(np.int8)
            b1_hat = (y.imag < 0).astype(np.int8)

            total_errs += (np.sum(b0_hat != bits[:, :, 0])
                           + np.sum(b1_hat != bits[:, :, 1]))
            total_bits += 2 * B * n_sym

        ber[k] = total_errs / total_bits

    return ber

# Closed-form reference  (should match MC as n_sym -> inf)
def ber_closed_form(h_true_all, h_pred_all, snr_db_range):
    """Analytical QPSK BER under MRT — same physics, zero sampling noise."""
    def _unit(H):
        n = np.linalg.norm(H, axis=1, keepdims=True)
        return H / np.maximum(n, 1e-30)
    ht, hp = _unit(h_true_all), _unit(h_pred_all)
    rho = np.abs(np.sum(np.conj(ht) * hp, axis=1)) ** 2   # |g|^2
    ber = np.zeros(len(snr_db_range))
    for k, snr_db in enumerate(snr_db_range):
        snr_lin = 10.0 ** (snr_db / 10.0)
        ber[k] = np.mean(0.5 * erfc(np.sqrt(rho * snr_lin / 2.0)))
    return ber

snr_range = np.arange(-5, 26, 1)

print(f"Running Monte Carlo BER  (N = {len(h_true)}, n_sym = 1000) ...")
ber_mc_perfect  = ber_monte_carlo(h_true, h_true, snr_range, n_sym=1000, seed=0)
ber_mc_baseline = ber_monte_carlo(h_true, h_base, snr_range, n_sym=1000, seed=1)
ber_mc_mmn      = ber_monte_carlo(h_true, h_mmn,  snr_range, n_sym=1000, seed=2)
ber_mc_ft       = ber_monte_carlo(h_true, h_ft,   snr_range, n_sym=1000, seed=3)

# Closed-form for validation (should lie right on top of MC curves)
ber_cf_perfect  = ber_closed_form(h_true, h_true, snr_range)
ber_cf_baseline = ber_closed_form(h_true, h_base, snr_range)
ber_cf_mmn      = ber_closed_form(h_true, h_mmn,  snr_range)
ber_cf_ft       = ber_closed_form(h_true, h_ft,   snr_range)

#  tabulate 
print(f"\n{'SNR':>5} | {'MC Perf':>9} | {'MC Base':>9} | {'MC MMN':>9} "
      f"| {'MC FT':>9} | {'CF Base':>9} | {'CF MMN':>9}")
print("-" * 72)
for i, snr in enumerate(snr_range):
    if snr % 5 == 0:
        print(f"{snr:5.0f} | {ber_mc_perfect[i]:9.2e} | {ber_mc_baseline[i]:9.2e} "
              f"| {ber_mc_mmn[i]:9.2e} | {ber_mc_ft[i]:9.2e} "
              f"| {ber_cf_baseline[i]:9.2e} | {ber_cf_mmn[i]:9.2e}")

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 9.5,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'lines.linewidth': 1.5,
    'lines.markersize': 5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'text.usetex': False,
})

def _safe(b):
    out = b.astype(float).copy()
    out[out <= 0] = np.nan
    return out

# Crop to -5..20 dB to avoid the MC sampling floor region
snr_plot = snr_range[snr_range <= 20]
idx = np.isin(snr_range, snr_plot)

fig, ax = plt.subplots(figsize=(7, 4.8))

# Closed-form as solid lines (the authoritative curves)
ax.semilogy(snr_plot, _safe(ber_cf_perfect[idx]),  'k-',  lw=1.5,
            label='Perfect CSI (bound)')
ax.semilogy(snr_plot, _safe(ber_cf_ft[idx]),       '-',   lw=1.5, color='#1D9E75',
            label='T-LSTM + 50-shot FT')
ax.semilogy(snr_plot, _safe(ber_cf_mmn[idx]),      '-',   lw=1.5, color='#7F77DD',
            label='T-LSTM + MetaModelNet')
ax.semilogy(snr_plot, _safe(ber_cf_baseline[idx]), '-',   lw=1.5, color='#E24B4A',
            label='T-LSTM (no adaptation)')

# MC markers overlaid for validation (every 2nd point to avoid clutter)
step = 2
ax.semilogy(snr_plot[::step], _safe(ber_mc_perfect[idx][::step]),
            'ko', ms=4.5, mfc='white', mew=1.2, zorder=5)
ax.semilogy(snr_plot[::step], _safe(ber_mc_ft[idx][::step]),
            's', ms=4.5, mfc='white', mew=1.2, color='#1D9E75', zorder=5)
ax.semilogy(snr_plot[::step], _safe(ber_mc_mmn[idx][::step]),
            '^', ms=4.5, mfc='white', mew=1.2, color='#7F77DD', zorder=5)
ax.semilogy(snr_plot[::step], _safe(ber_mc_baseline[idx][::step]),
            'd', ms=4.5, mfc='white', mew=1.2, color='#E24B4A', zorder=5)

ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Bit Error Rate')
ax.set_xlim([-5, 20])
ax.set_ylim([1e-5, 5e-1])
ax.grid(True, which='major', alpha=0.25, linewidth=0.6)
ax.grid(True, which='minor', alpha=0.12, linewidth=0.4)
ax.legend(loc='lower left', framealpha=0.9, edgecolor='0.8')

plt.tight_layout(pad=0.4)
plt.savefig('ber_vs_snr_report.png', dpi=300, bbox_inches='tight')
plt.savefig('ber_vs_snr_report.pdf', bbox_inches='tight')
plt.show()
print("Saved: ber_vs_snr_report.png / .pdf")


# ACHIEVABLE RATE  (same alignment factor, same normalisation)
def compute_rate(h_true_all, h_pred_all, snr_db_range):
    def _unit(H):
        n = np.linalg.norm(H, axis=1, keepdims=True)
        return H / np.maximum(n, 1e-30)
    ht, hp = _unit(h_true_all), _unit(h_pred_all)
    rho = np.abs(np.sum(np.conj(ht) * hp, axis=1)) ** 2
    rate = np.zeros(len(snr_db_range))
    for k, snr_db in enumerate(snr_db_range):
        snr_lin = 10.0 ** (snr_db / 10.0)
        rate[k] = np.mean(np.log2(1.0 + rho * snr_lin))
    return rate

rate_perfect  = compute_rate(h_true, h_true, snr_range)
rate_baseline = compute_rate(h_true, h_base, snr_range)
rate_ft       = compute_rate(h_true, h_ft,   snr_range)
rate_mmn      = compute_rate(h_true, h_mmn,  snr_range)

print(f"\n{'SNR':>5} | {'Perfect':>8} | {'Baseline':>8} | {'MMN':>8} "
      f"| {'50-FT':>8} | {'MMN gain':>8}")
print("-" * 62)
for i, snr in enumerate(snr_range):
    if snr % 5 == 0:
        print(f"{snr:5.0f} | {rate_perfect[i]:8.3f} | {rate_baseline[i]:8.3f} | "
              f"{rate_mmn[i]:8.3f} | {rate_ft[i]:8.3f} | "
              f"{rate_mmn[i] - rate_baseline[i]:8.3f}")